<a href="https://colab.research.google.com/github/Gab-San/lbm-simulation-lib/blob/main/notebooks/cuda_all_simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LBM CUDA: build and run directly from a ZIP on Colab

Use this notebook when you want Colab to compile **exactly the ZIP you upload**, without cloning GitHub. This avoids stale branches and stale `/content/lbm-simulation-lib` directories.

Select **Runtime → Change runtime type → GPU** before running the notebook.


In [ ]:
!nvcc --version
!nvidia-smi


## Upload the repository ZIP

Upload `lbm-simulation-lib-sim-cuda-simulations-fixed.zip` (or a later ZIP with the same project layout).


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile
import os

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one repository ZIP"
zip_name = next(iter(uploaded))
zip_path = Path('/content') / zip_name

work = Path('/content/lbm_uploaded')
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(work)

# Find the project root by CMakeLists.txt + simulations/cuda.
candidates = [p.parent for p in work.rglob('CMakeLists.txt') if (p.parent/'simulations/cuda').is_dir()]
assert len(candidates) == 1, f"Expected one project root, found: {candidates}"
repo = candidates[0]
print('Repository:', repo)
os.chdir(repo)
%cd {repo}


In [ ]:
from pathlib import Path

required = [
    Path("include/lbm-sim/collision-operators/collision-strategy.hpp"),
    Path("include/lbm-sim/solver/cuda-solver.cuh"),
    Path("simulations/cuda/lid_cavity_2d_bgk.cu"),
    Path("simulations/cuda/lid_cavity_2d_trt.cu"),
    Path("simulations/cuda/couette_flow_2d_bgk.cu"),
    Path("simulations/cuda/couette_flow_2d_trt.cu"),
    Path("simulations/cuda/poiseuille_flow_2d_bgk.cu"),
    Path("simulations/cuda/poiseuille_flow_2d_trt.cu"),
]
missing = [str(x) for x in required if not x.exists()]
assert not missing, f"Wrong project layout; missing: {missing}"

# The obsolete header must not be referenced by this revision.
refs = []
for p in list(Path('include').rglob('*')) + list(Path('simulations').rglob('*')):
    if p.is_file() and p.suffix in {'.hpp','.cuh','.cpp','.cu'}:
        if 'collision-strategy-cuda.cuh' in p.read_text(errors='ignore'):
            refs.append(str(p))
assert not refs, f"Obsolete collision-strategy-cuda.cuh reference found in: {refs}"
print('Source revision/layout OK')


In [ ]:
import subprocess, shutil

try:
    cap = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()[0]
    CUDA_ARCH = cap.replace(".", "")
except Exception:
    CUDA_ARCH = "75"

print("CMAKE_CUDA_ARCHITECTURES =", CUDA_ARCH)
shutil.rmtree('build', ignore_errors=True)
shutil.rmtree('logs', ignore_errors=True)
Path('out').mkdir(exist_ok=True)

!cmake -S . -B build -DLBM_ENABLE_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}
!cmake --build build -j2
!find build/simulations -maxdepth 1 -type f -executable -printf "%f\n" | sort


## Run all six CUDA executables

This runs Couette BGK/TRT, Poiseuille BGK/TRT, and lid-cavity BGK/TRT. Note that each lid-cavity executable also contains the 2000×2000 and 5000×5000 cases, so this cell is intentionally a **full run**, not a quick smoke test.


In [ ]:
!chmod +x run_cuda_simulations.sh
!./run_cuda_simulations.sh


## Validate outputs


In [ ]:
!python scripts/validate_outputs.py


In [7]:
%pip -q install -r scripts/requirements.txt


## Comparison plots


In [ ]:
!python scripts/visualize_profile.py \
  out/data_couette_cuda_129_100_01_bgk.bin \
  out/data_couette_cuda_129_100_01_trt.bin \
  -o out/couette_cuda_129_100_01_bgk_vs_trt.png \
  --title="CUDA Couette: BGK vs TRT" --xlabel="y"

!python scripts/visualize_profile.py \
  out/data_poiseuille_cuda_129_100_01_bgk.bin \
  out/data_poiseuille_cuda_129_100_01_trt.bin \
  -o out/poiseuille_cuda_129_100_01_bgk_vs_trt.png \
  --title="CUDA Poiseuille: BGK vs TRT" --xlabel="y"

!python scripts/visualize_profile.py \
  out/data_lid_cavity_cuda_129_100_bgk.bin \
  out/data_lid_cavity_cuda_129_100_trt.bin \
  benchmarks/ghia/data_y_100.txt \
  -o out/lid_cavity_cuda_129_re100_vs_ghia.png \
  --title="CUDA Lid Cavity Re=100: BGK/TRT vs Ghia" --xlabel="x"


## Download results as one ZIP


In [10]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/lbm_cuda_outputs', 'zip', root_dir='.', base_dir='out')
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>